# Exploring eBay Car sales data
For this project we will use this 	[dataset](https://data.world/data-society/used-cars-data).
The aim of this project is to clean the data and analyze the included used car listings.

- `dateCrawled` - When this ad was first crawled. All field-values are taken from this date.
- `name` - Name of the car.
- `seller` - Whether the seller is private or a dealer.
- `offerType` - The type of listing
- `price` - The price on the ad to sell the car.
- `abtest` - Whether the listing is included in an A/B test.
- `vehicleType` - The vehicle Type.
- `yearOfRegistration` - The year in which the car was first registered.
- `gearbox` - The transmission type.
- `powerPS` - The power of the car in PS.
- `model` - The car model name.
- `kilometer` - How many kilometers the car has driven.
- `monthOfRegistration` - The month in which the car was first registered.
- `fuelType` - What type of fuel the car uses.
- `brand` - The brand of the car.
- `notRepairedDamage` - If the car has a damage which is not yet repaired.
- `dateCreated` - The date on which the eBay listing was created.
- `nrOfPictures` - The number of pictures in the ad.
- `postalCode` - The postal code for the location of the vehicle.
- `lastSeenOnline` - When the crawler saw this ad last online.

In [ ]:
import pandas as pd
import numpy as np
autos = pd.read_csv('autos.csv', encoding='Latin-1')
autos.info()
autos.head()

### Clean columns

We can see that most of the columns are strings; some columns have null values; columns use camelcase instead of Python's preferred snakecase. We will change columns names to use more appropriate names.

In [ ]:

autos.columns = ['date_crawled', 'name', 'seller', 'offer_type', 'price', 'ab_test',
       'vehicle_type', 'registration_year', 'gearbox', 'power_ps', 'model',
       'odometer', 'registration_month', 'fuel_type', 'brand',
       'unrepaired_damage', 'ad_created', 'num_photos', 'postal_code',
       'last_seen']
autos.head()

### Data exploration and cleaning

By exploring the data we can find obvious areas to clean

In [ ]:
autos.describe(include='all')

In [ ]:
autos

We can observe here that two columns have almost unique value inside: `seller` and `offer_type`. The `price` and `odometer`columns are numeric values stored as text.
Also, the `num_photos`column look strange.

In [ ]:
autos['price'] = autos['price'].str.replace('$','').str.replace(',','').astype(int)

In [ ]:
autos['odometer'] = autos['odometer'].str.replace('km','').str.replace(',','').astype(int)
autos.rename({'odometer':'odometer_km'},axis=1,inplace=True)

In [ ]:
autos['num_photos'].value_counts()

The `num_photos` column is useless, we are going to drop it, as well as `seller` and `offer_type`. 

In [ ]:
autos = autos.drop(['num_photos','seller','offer_type'], axis=1)

### Exploring Odometer and Price

In [ ]:
autos['odometer_km'].value_counts()

In [ ]:
print(autos['price'].unique().shape[0], ' unique values')
print(autos['price'].describe())
print(autos['price'].value_counts().head(20))

1421 cars are listed with a 0 dollar price. Because that's less than 2% of all the raws, we can remove them. Also, there seems to be a problem with the highest price, at 100 millions dollars.

In [ ]:
autos['price'].value_counts().sort_index(ascending=False).head(20)

In [ ]:
autos['price'].value_counts().sort_index(ascending=True).head(20)

There is 14 rows with a price of more than 350k dollars. We will get rid of these less realistic rows, as well as 0 dollar rows.

In [ ]:
autos = autos[autos['price'].between(1,350001)]
autos['price'].describe()

### Exploring dates

We will explore these columns, which contains date informations.
- `date_crawled`
- `registration_month`
- `registration_year`
- `ad_created`
- `last_seen`

In [ ]:
(autos['date_crawled']
     .str[:10]
     .value_counts(normalize=True, dropna=False) # % instead of count
     .sort_values()
)

It looks like the website was crawled daily for a month between march 2016 and april 2016. The distribution of crawling looks equivalent.

In [ ]:
(autos['last_seen']
     .str[:10]
     .value_counts(normalize=True, dropna=False)
     .sort_values(ascending=False)
)

This serie indicate us the last time the crawler has seen an ad, which should correspond to the date of selling. We can observe a very high number of `last_seen` values during the last days, 6 to 10 times the values from the previous days. This indicate the end of the crawling period, not a huge rise in sales.

In [ ]:
(autos['ad_created']
     .str[:10]
     .value_counts(normalize=True, dropna=False)
     .sort_index(ascending=False)
)

The majority of ads posted on the website were created during the crawling period. But we can also observe old ads, created several months before the crawling period.

In [ ]:
autos['registration_year'].describe()

The `registration_year` value indicate the first registration of the car, which mean the age of the car. We have some odd values, like the maximum of year 9999

### Cleaning registration year

First we can delete rows with a registration year after 2016, which is the year of the listing. For the early registration year, we are gonna delete values before 1900.

In [ ]:
percentage = round(1-(autos['registration_year'].between(1900,2016).sum() / autos.shape[0]),4)
print('percentage of incorrect values: ', percentage*100, '%')

As these incorrect values represent less than 4% of the dataset, we can erase them.

In [ ]:
autos = autos[autos['registration_year'].between(1900,2016)]
autos['registration_year'].value_counts(normalize=True).head(10)

We can observe that after the cleaning for the registration year values, most of the car wre registered in the last 20 years.

### Exploring price by brand

In [ ]:
brand_counts = autos['brand'].value_counts(normalize=True).head(20)
brand_counts

We can observe that German brands represents more than 50% of the top 20 car brands in this dataset.
We will focus on the the brands representing more than 5% of the dataset because there is a lot of brands with very small representation on the dataset.

In [ ]:
top_brands = brand_counts[brand_counts > 0.05].index
top_brands

In [ ]:
brand_mean_price = {} # store the mean price with the brand

for b in top_brands:
    br = autos[autos['brand'] == b] # create a filtered dataset 
    mean_price = br['price'].mean()
    brand_mean_price[b] = int(mean_price)

brand_mean_price

In [ ]:
brand_mean_mileage = {}

for b in top_brands:
    m = autos[autos['brand'] == b]
    mean_mileage = m['odometer_km'].mean()
    brand_mean_mileage[b] = int(mean_mileage)
brand_mean_mileage

In [ ]:
mean_price = pd.Series(brand_mean_price) # create pandas serie from dictionnary
mean_mileage = pd.Series(brand_mean_mileage)

brand_info = pd.DataFrame(mean_price, columns=['mean_price']) # create a dataframe from one serie
brand_info['mileage'] = mean_mileage # add other serie to new dataset
brand_info.sort_values('mean_price', ascending=False)

For each top brands, the price can vary a lot but not the mileage. We can observe a trend: high priced cars tend to have a higher mileage, and lower priced cars tend to have a lower mileage.